In [ ]:
# ==============================================================================
# SEL 1: Cek GPU Colab (Memastikan GPU T4 Aktif)
# ==============================================================================
!nvidia-smi

In [ ]:
# ==============================================================================
# SEL 2: Install Depedensi AI (Transformers, PEFT, BitsAndBytes, TRL)
# ==============================================================================
# CATATAN PENTING soal versi library:
# - transformers/trl/peft/accelerate DIKUNCI (bukan -U) karena versi
#   terbaru sempat memicu 3 bug kode berbeda (SFTConfig vs TrainingArguments,
#   rename tokenizer->processing_class, bug internal _patch_chunked_ce_lm_head).
# - bitsandbytes SENGAJA TIDAK DIKUNCI (biarkan versi terbaru terinstall).
#   Alasan beda dari yang di atas: versi lama (0.44.1) yang sempat dikunci
#   TIDAK PUNYA file biner CUDA yang cocok dengan environment Kaggle/Colab
#   (CUDA 12.8 di 2026) -- errornya "Could not find bitsandbytes CUDA
#   binary" lalu jatuh ke jalur cadangan triton yang JUGA rusak (triton.ops
#   tidak ada lagi di versi triton terbaru). Ini soal kecocokan hardware/
#   CUDA runtime, beda akar masalah dari bug API trl -- solusinya kebalikan:
#   biarkan bitsandbytes ikut versi terbaru yang sudah punya biner CUDA
#   cocok untuk environment saat ini.
!pip install -q transformers==4.46.3 datasets peft==0.13.2 accelerate==0.34.2 trl==0.12.2 huggingface_hub
!pip install -q -U bitsandbytes

In [ ]:
# ==============================================================================
# SEL 3: Clone Repositori GitHub SCED Engine & Load Data
# ==============================================================================
import os

# Clone repo GitHub 4IGen
REPO_URL = "https://github.com/andiagilrachman/4IGen-x-SCED-Engine-Master-Blueprint-Document.git"
REPO_DIR = "4IGen-x-SCED-Engine"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Simpan path ABSOLUT (bukan cuma nama relatif) supaya sel-sel selanjutnya
# (terutama Sel 9 & 10) bisa selalu balik ke folder yang benar lewat
# os.chdir(REPO_ABS_PATH), tidak bergantung pada working directory yang
# "diwariskan" antar sel (bisa geser kalau ada sel yang di-run ulang manual
# di luar urutan, atau sesi Kaggle/Colab yang panjang).
REPO_ABS_PATH = os.getcwd()
print(f"Repositori berhasil di-clone! Path absolut: {REPO_ABS_PATH}")

In [ ]:
# ==============================================================================
# SEL 4: Load Base Model (Qwen 2.5 7B Instruct) dengan Kuantisasi 4-bit
# ==============================================================================
import os
# PENTING: sembunyikan GPU kedua SEBELUM import torch. device_map={"": 0}
# saja TIDAK CUKUP -- itu cuma atur penempatan BOBOT model, tapi Trainer
# tetap mendeteksi 2 GPU tersedia (Kaggle sediakan T4 x2) dan otomatis
# coba bungkus model dengan nn.DataParallel, yang ternyata tidak cocok
# dengan model quantized+PEFT kita (RuntimeError "chunk expects at least
# a 1-dimensional tensor" saat scatter input ke 2 GPU). Fix yang benar:
# batasi visibilitas GPU dari awal, SEBELUM torch di-import, supaya
# torch.cuda.device_count() melaporkan 1 (bukan 2) dan Trainer tidak
# pernah mempertimbangkan DataParallel sama sekali.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# CATATAN: pakai bfloat16 (bukan float16) untuk compute_dtype & torch_dtype.
# Alasan: checkpoint asli Qwen 2.5 memang tersimpan dalam bfloat16, dan
# fp16=True (grad scaler) TIDAK KOMPATIBEL dengan bfloat16 sama sekali
# (NotImplementedError saat trainer.train() -- sudah dicoba paksa cast
# manual ke float16 tapi tetap gagal karena ada bagian lain di training
# loop yang tetap hasilkan gradien bfloat16). Solusi tepatnya: ikuti
# dtype asli checkpoint (bfloat16) secara konsisten dari awal, dan pakai
# bf16=True di Sel 7 (BUKAN fp16=True) -- bf16 tidak butuh grad scaler
# sama sekali sehingga masalah ini hilang dari akarnya. GPU T4 tidak
# punya akselerasi hardware khusus bf16 (sedikit lebih lambat dari fp16),
# tapi tetap berfungsi normal.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print("Base Model Qwen 2.5 7B terkonfigurasi 4-bit berhasil dimuat!")

In [ ]:
# ==============================================================================
# SEL 5: Konfigurasi LoRA Adapter (SCED Adapter Config)
# ==============================================================================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

# Pastikan semua parameter yang bisa dilatih (adapter LoRA) konsisten
# bfloat16, sesuai keputusan di Sel 4 (lihat catatan di sana soal alasan
# pindah dari float16 ke bfloat16 -- masalah GradScaler tidak kompatibel
# dengan bfloat16 diselesaikan dengan TIDAK PAKAI GradScaler sama sekali
# lewat bf16=True di Sel 7, bukan lagi memaksa semuanya ke float16).
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

model.print_trainable_parameters()

In [ ]:
# ==============================================================================
# SEL 6: Siapkan Dataset Latihan
# ==============================================================================
from datasets import load_dataset

DATASET_PATH = "data/training_jsonl/sced_scaled_train_v1.jsonl"
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_prompts(batch):
    texts = []
    for messages in batch["messages"]:
        # Mengubah format pesan menjadi format percakapan resmi Qwen
        formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(formatted_chat)
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)
print(f"Total Sampel Tersedia: {len(formatted_dataset)}")

# ==============================================================================
# MODE SMOKE TEST: GPU T4 + bf16 (wajib, lihat catatan Sel 4) sangat lambat,
# training penuh (684 sampel x 3 epoch) bisa 10+ jam -- berisiko sesi Colab
# putus sebelum selesai. Untuk BUKTIKAN dulu pipeline lengkap (Sel 7-10)
# jalan benar, dataset dipangkas jadi subset kecil di sini.
#
# SET SMOKE_TEST_MODE = False lalu jalankan ulang Sel 6 ini untuk training
# PENUH pakai semua data (setelah smoke test terbukti berhasil, idealnya
# di sesi Colab yang lebih panjang / Colab Pro).
# ==============================================================================
SMOKE_TEST_MODE = False  # DIMATIKAN -- smoke test terbukti pipeline jalan, sekarang training PENUH
SMOKE_TEST_SAMPLE_SIZE = 60

if SMOKE_TEST_MODE:
    formatted_dataset = formatted_dataset.shuffle(seed=42).select(
        range(min(SMOKE_TEST_SAMPLE_SIZE, len(formatted_dataset)))
    )
    print(f"SMOKE TEST MODE AKTIF: dataset dipangkas ke {len(formatted_dataset)} sampel "
          f"(dari total yang tersedia di atas) supaya training selesai dalam waktu wajar.")

In [ ]:
# ==============================================================================
# SEL 7: Pengaturan Training (SFTTrainer) & Eksekusi Fine-Tuning
# ==============================================================================
from trl import SFTTrainer, SFTConfig

# CATATAN: trl versi terbaru memindahkan dataset_text_field & max_seq_length
# dari SFTTrainer langsung ke SFTConfig (dulu pakai TrainingArguments biasa).
# Kode di bawah dibuat tahan terhadap 2 variasi API trl yang pernah ada.
#
# PENTING (pelajaran dari kejadian sesi Kaggle reset di tengah jalan,
# adapter hasil training PENUH sempat hilang total karena cuma disimpan
# di akhir): save_strategy sekarang "steps" dengan save_steps=25, jadi
# checkpoint tersimpan BERKALA ke disk selama training berlangsung, bukan
# cuma sekali di akhir. Kalau sesi reset di tengah jalan lagi, progres
# checkpoint terakhir tidak hilang total.
sft_config_kwargs = dict(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,  # BUKAN fp16=True -- lihat catatan di Sel 4 soal alasannya
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,  # cuma simpan 2 checkpoint terbaru, hemat disk
    report_to="none",
    dataset_text_field="text",
)

try:
    sft_config = SFTConfig(max_seq_length=1024, **sft_config_kwargs)
except TypeError:
    # versi trl yang lebih baru lagi mengganti nama jadi max_length
    sft_config = SFTConfig(max_length=1024, **sft_config_kwargs)

try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        args=sft_config,
        processing_class=tokenizer,
    )
except TypeError:
    # versi trl yang lebih lama masih pakai nama parameter "tokenizer"
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        args=sft_config,
        tokenizer=tokenizer,
    )

print("Memulai Training SCED Engine v0.1...")
trainer.train()
print("Training Selesai!")

# PENTING: begitu training selesai, LANGSUNG lanjut Sel 8 lalu Sel 10
# (zip & download) TANPA JEDA -- jangan buka tab lain / biarkan idle lama
# dulu sebelum adapter benar-benar terunduh ke PC. Sel 9 (ujian ICBP) bisa
# ditunda setelah adapter aman terdownload.

In [ ]:
# ==============================================================================
# SEL 8: Simpan Adapter SCED Engine v0.1
# ==============================================================================
OUTPUT_ADAPTER_DIR = "models/sced_adapter_v0.1"
model.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)
print(f"Adapter berhasil disimpan di: {OUTPUT_ADAPTER_DIR}")

In [ ]:
# ==============================================================================
# SEL 9: Ujian Kelulusan (Testing Unseen Data ICBP)
# ==============================================================================
import json
import os

# Pastikan working directory benar (kembali ke folder repo lewat path
# ABSOLUT dari Sel 3) -- jaga-jaga kalau working directory "geser" di
# antara sel (bisa terjadi kalau ada sel yang di-run ulang manual di luar
# urutan, atau sesi Kaggle/Colab yang sudah berjalan lama).
os.chdir(REPO_ABS_PATH)

# Read Soal Ujian ICBP
with open("eval/icbp_test_schema.json", "r") as f:
    icbp_data_full = json.load(f)

# PENTING: samakan struktur input dengan format training di sced_pilot_train.jsonl
# Training hanya menyertakan "financial_metrics" + "macro_context" (metadata DIBUANG).
# Jika "metadata" ikut disertakan saat eval, struktur input berbeda dari saat training
# sehingga hasil ujian ICBP tidak representatif untuk mengukur generalisasi model.
icbp_data_eval = {
    "financial_metrics": icbp_data_full["financial_metrics"],
    "macro_context": icbp_data_full["macro_context"]
}

prompt_user = f"""[LENSA]: Value & Risk Margin
[ASET]: ICBP
[INSTRUCTION]: Sebagai investor yang berorientasi pada nilai dan keamanan modal, bagaimana Anda menilai profil imbal-hasil dan risiko ICBP?

[INPUT DATA]: {json.dumps(icbp_data_eval)}"""

eval_messages = [
    {"role": "system", "content": "Kamu adalah SCED Engine, AI Finansial & Ekonomi di 4IGen.com. Tugasmu adalah memberikan analisis berdasarkan Data Schema Murni yang diberikan dengan metode Chain-of-Thought (CoT) 4 langkah. Dilarang mengarang angka di luar data input."},
    {"role": "user", "content": prompt_user}
]

prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=700,
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("="*60)
print("HASIL ANALISIS SOAL UJIAN (ICBP) OLEH SCED ENGINE v0.1:")
print("="*60)
print(response)

In [ ]:
# ==============================================================================
# SEL 10: Zip Adapter untuk Download ke PC Rumah
# ==============================================================================
import os

# Pastikan working directory benar (kembali ke folder repo lewat path
# ABSOLUT dari Sel 3) -- ini penyebab error sebelumnya: "zip warning: name
# not matched: models/sced_adapter_v0.1" karena working directory sempat
# geser dari folder repo.
os.chdir(REPO_ABS_PATH)

!zip -r sced_adapter_v0.1.zip models/sced_adapter_v0.1

# Deteksi environment: PENTING deteksi Kaggle secara eksplisit dulu (cek
# folder /kaggle) SEBELUM coba import google.colab -- ditemukan Kaggle
# ternyata punya semacam stub/placeholder modul "google.colab" yang bisa
# di-import TANPA ImportError meski bukan Colab sungguhan, jadi deteksi
# try/except ImportError saja tidak cukup diandalkan dan sempat salah
# masuk jalur Colab padahal di Kaggle.
IS_KAGGLE = os.path.exists("/kaggle")

if not IS_KAGGLE:
    try:
        from google.colab import files
        files.download("sced_adapter_v0.1.zip")
        print("File adapter terunduh otomatis (Colab)! Siap dipindahkan ke PC rumah.")
    except ImportError:
        IS_KAGGLE = True  # anggap platform lain non-Colab, treatment sama seperti Kaggle di bawah

if IS_KAGGLE:
    full_path = os.path.abspath("sced_adapter_v0.1.zip")
    print(f"File adapter siap: {full_path}")
    print("Platform ini (Kaggle atau non-Colab lain) tidak mendukung download otomatis.")
    print("Cara download manual:")
    print("- Kaggle: buka panel 'Output' di sisi kanan notebook, cari")
    print("  sced_adapter_v0.1.zip, klik ikon download di sampingnya.")
    print("- Platform lain: cari file tsb di file browser sisi kiri, klik")
    print("  kanan -> Download.")